# Colab H3 图生视频部署

本 Notebook 使用 ComfyUI 与 MiniMax H3 官方权重。5 秒和 10 秒工作流在 A100 80GB 上成功执行过；本公开副本已清除所有运行输出、账号信息和授权痕迹。官方 Colab CLI SSH 隧道、Mac REST/CORS/WebSocket 已验证；用户报告画布生成成功，但该次任务的 MP4 规格、音轨与实际提交工作流尚未独立核验。

依次运行环境自检、Drive 挂载、安装恢复、启动、节点检查、参考图、工作流和 REST 客户端。5 秒回归测试可选；直接运行 10 秒会消耗 GPU 计算单元。Drive 与 Colab CLI 授权由账号持有人自行完成。模型在 Colab 本地盘运行，Drive 只用于持久化。ComfyUI 只监听运行时的 `127.0.0.1:8188`。


In [ ]:
# ============ 单元 1：环境自检 ============
import os, sys, shutil, subprocess

print("=" * 68)
print("Colab-H3 环境自检")
print("=" * 68)

ok = True

# --- GPU ---
gpu_name, vram = None, 0.0
if shutil.which("nvidia-smi"):
    raw = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True).stdout.strip()
    print("GPU        :", raw)
    gpu_name = raw.split(",")[0].strip()
    try:
        vram = float(raw.split(",")[1].replace("MiB", "").strip()) / 1024
    except Exception:
        pass
else:
    print("GPU        : 未检测到")
    ok = False

# --- torch ---
try:
    import torch
    print("torch      :", torch.__version__, "| CUDA:", torch.version.cuda,
          "| 可用:", torch.cuda.is_available())
    if torch.cuda.is_available() and not vram:
        vram = torch.cuda.get_device_properties(0).total_memory / 2**30
except ImportError:
    print("torch      : 未安装")

print("显存       : %.1f GB" % vram)

if vram and vram < 20:
    ok = False
    print("\n  [不可行] 显存 < 20 GB。H3 剪枝底模 19.5 GB 无法加载。")
    print("           请到「代码执行程序 → 更改运行时类型」改选 L4 或 A100，再重跑本格。")

# --- 系统内存 ---
ram = 0.0
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                ram = int(line.split()[1]) / 1024**2
                break
except Exception:
    pass
print("系统内存   : %.1f GB" % ram)
if ram and ram < 16:
    print("  [注意] 内存偏低。--disable-pinned-memory 已启用，但仍可能被 host OOM 杀掉。")

# --- 磁盘 ---
du = shutil.disk_usage("/content")
print("本地磁盘   : 可用 %.1f GB / 共 %.1f GB" % (du.free / 2**30, du.total / 2**30))

print("=" * 68)
print("结论:", "可以继续" if ok else "先解决上面的问题")

In [ ]:
# ============ 单元 2：目录约定（权重走本地盘，Drive 用于备份）============
# 说明：权重 41.4 GB 放本地盘，读取比 Drive 快得多；Drive 保存模型、产物和日志。
#       Drive 挂不上 / 空间不够也不影响部署，只是产物不能持久化。
import os, shutil

LOCAL_ROOT = "/content/ColabH3"
COMFY_DIR  = "/content/ComfyUI"
BASE       = "http://127.0.0.1:8188"
MODELS_DIR = LOCAL_ROOT + "/models"
OUT_DIR    = LOCAL_ROOT + "/outputs"
LOG_DIR    = LOCAL_ROOT + "/logs"
HF_CACHE   = LOCAL_ROOT + "/hf_cache"
for d in (LOCAL_ROOT, MODELS_DIR, OUT_DIR, LOG_DIR, HF_CACHE):
    os.makedirs(d, exist_ok=True)

DRIVE_ROOT = None
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if os.path.exists("/content/drive/MyDrive"):
        DRIVE_ROOT = "/content/drive/MyDrive/Colab-H3"
        for sub in ("outputs", "workflows", "logs"):
            os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
        free = shutil.disk_usage("/content/drive/MyDrive").free / 2**30
        print("Drive 可用 : %.1f GB" % free)
        print("Drive 目录 :", DRIVE_ROOT)
    else:
        print("Drive 挂载未完成，本次产物只留在本地")
except Exception as e:
    print("Drive 不可用（不影响部署）:", repr(e)[:200])

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

du = shutil.disk_usage("/content")
print()
print("本地盘可用 : %.1f GB" % (du.free / 2**30))
print("权重目录   :", MODELS_DIR)
print("产物目录   :", OUT_DIR)
print("日志        :", LOG_DIR)


In [ ]:
# 单元 3：复用现有 ComfyUI；新环境安装到已实测提交
import os, subprocess, sys
COMFY_COMMIT = "4d64a8bc072b81c9b47321789b4d83fb7348ae2a"
ESSENTIALS_COMMIT = "9d9f4bedfc9f0321c19faf71855e228c93bd0dc9"

def clone_at_commit(url, path, commit):
    subprocess.run(["git", "clone", "--depth", "1", url, path], check=True)
    subprocess.run(["git", "-C", path, "fetch", "--depth", "1", "origin", commit], check=True)
    subprocess.run(["git", "-C", path, "checkout", "--detach", commit], check=True)

if not os.path.isdir(COMFY_DIR):
    clone_at_commit("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, COMFY_COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", os.path.join(COMFY_DIR, "requirements.txt")], check=True)
else:
    print("复用现有 ComfyUI：", COMFY_DIR)
ess = os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI_essentials")
if not os.path.isdir(ess):
    clone_at_commit("https://github.com/cubiq/ComfyUI_essentials.git", ess, ESSENTIALS_COMMIT)
    req = os.path.join(ess, "requirements.txt")
    if os.path.isfile(req):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", req], check=True)
for sub in ("diffusion_models", "text_encoders", "vae", "loras"):
    os.makedirs(os.path.join(MODELS_DIR, sub), exist_ok=True)
    os.makedirs(os.path.join(COMFY_DIR, "models", sub), exist_ok=True)
print("ComfyUI:", subprocess.getoutput("git -C /content/ComfyUI rev-parse --short HEAD"))


In [ ]:
# 单元 4：优先恢复 Drive 中完整权重，再从官方仓库补齐
import os, time, shutil
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub'], check=True)
    from huggingface_hub import hf_hub_download
REPO = "Comfy-Org/MiniMax-H3"
MODEL_REVISION = "54472221d27c695fca9621b0c3ed52f06c4af707"
FILES = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", 19.53),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", 14.61),
    ("vae/minimax_h3_video_vae_fp16.safetensors", 4.85),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", 0.564),
    ("loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors", 1.822),
]
def complete(path, gib):
    return os.path.isfile(path) and os.path.getsize(path) >= gib * 2**30 * 0.97
for rel, gib in FILES:
    dst = os.path.join(MODELS_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not complete(dst, gib):
        saved = os.path.join(DRIVE_ROOT, "models", rel) if DRIVE_ROOT else None
        if saved and complete(saved, gib):
            print("从 Drive 恢复：", rel, flush=True)
            shutil.copy2(saved, dst)
        else:
            print("下载官方权重：", rel, flush=True)
            hf_hub_download(repo_id=REPO, filename=rel, revision=MODEL_REVISION, local_dir=MODELS_DIR)
    assert complete(dst, gib), "权重不完整：" + rel
    link = os.path.join(COMFY_DIR, "models", rel)
    if not os.path.exists(link):
        os.symlink(dst, link)
    print("就绪：", rel)


In [ ]:
# 单元 5：复用或启动仅本机可访问的 ComfyUI
import os, subprocess, sys, time, requests
log_path = os.path.join(LOG_DIR, "comfyui.log")
CANVAS_ORIGIN = os.environ.get("CANVAS_ORIGIN", "http://127.0.0.1:3000")
def healthy():
    try:
        return requests.get(BASE + "/system_stats", timeout=5).status_code == 200
    except requests.RequestException:
        return False
if healthy():
    print("复用现有 ComfyUI：", BASE)
else:
    log = open(log_path, "ab")
    proc = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-pinned-memory", "--disable-api-nodes", "--enable-cors-header", CANVAS_ORIGIN], cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if healthy():
            break
        if proc.poll() is not None:
            raise RuntimeError("ComfyUI 退出：" + open(log_path, errors="replace").read()[-2000:])
        time.sleep(5)
assert healthy(), "ComfyUI 未就绪：" + open(log_path, errors="replace").read()[-2000:]
print("ComfyUI 版本：", requests.get(BASE + "/system_stats").json()["system"]["comfyui_version"])


In [ ]:
# ============ 单元 6：环境校验（节点 / 模型 / 显存）============
import json, requests

need_nodes = {"UNETLoader", "CLIPLoader", "VAELoader", "LoraLoaderModelOnly",
              "MiniMaxH3ImageToVideo", "ComfyMathExpression", "SamplerCustomAdvanced",
              "BasicScheduler", "CreateVideo", "SaveVideo", "VAEDecode", "VAEDecodeAudio",
              "LoadImage", "PrimitiveInt", "PrimitiveStringMultiline",
              "BasicGuider", "KSamplerSelect", "RandomNoise"}

info = requests.get(BASE + "/object_info", timeout=60).json()
have = set(info.keys())
missing = sorted(need_nodes - have)
print("节点总数      :", len(have))
print("缺失必需节点  :", missing if missing else "无")

if "MiniMaxH3ImageToVideo" in info:
    req = info["MiniMaxH3ImageToVideo"].get("input", {})
    print("H3 节点输入   :", list(req.get("required", {}).keys()))

# 模型是否被 ComfyUI 识别到
def options(node, field):
    try:
        return info[node]["input"]["required"][field][0]
    except Exception:
        return []

for node, field, want in [
    ("UNETLoader", "unet_name", "minimax_h3_fl2va_pruned_int8_convrot.safetensors"),
    ("CLIPLoader", "clip_name", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("VAELoader", "vae_name", "minimax_h3_video_vae_fp16.safetensors"),
    ("LoraLoaderModelOnly", "lora_name", "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"),
]:
    opts = options(node, field)
    print("%-22s %s -> %s" % (node, field, "OK" if want in opts else "未找到！"))

st = requests.get(BASE + "/system_stats", timeout=10).json()
for d in st.get("devices", []):
    print("设备: %s | 显存总量 %.1f GB | 空闲 %.1f GB" % (
        d.get("name"), d.get("vram_total", 0) / 2**30, d.get("vram_free", 0) / 2**30))
assert not missing, "缺少必需节点：" + ", ".join(missing)


In [ ]:
# 单元 7：上传参考图；可把两个路径改成自己上传的 PNG/JPG
from pathlib import Path
from PIL import Image, ImageDraw
FIRST_FRAME_PATH = "/content/h3_first.png"
LAST_FRAME_PATH = "/content/h3_last.png"
if not Path(FIRST_FRAME_PATH).exists():
    im = Image.new("RGB", (960, 544), (17, 24, 43))
    ImageDraw.Draw(im).ellipse((260, 170, 460, 370), fill=(44, 132, 224), outline=(171, 226, 255), width=8)
    im.save(FIRST_FRAME_PATH)
if not Path(LAST_FRAME_PATH).exists():
    im = Image.new("RGB", (960, 544), (34, 22, 46))
    ImageDraw.Draw(im).ellipse((500, 130, 730, 360), fill=(68, 171, 242), outline=(217, 246, 255), width=8)
    im.save(LAST_FRAME_PATH)
def upload_image(path):
    with open(path, "rb") as f:
        r = requests.post(BASE + "/upload/image", files={"image": (Path(path).name, f)}, data={"overwrite": "true"}, timeout=120)
    r.raise_for_status()
    return r.json()["name"]
n1, n2 = upload_image(FIRST_FRAME_PATH), upload_image(LAST_FRAME_PATH)
print("已上传：", n1, n2)


In [ ]:
# 单元 8：已在 A100 上执行成功的 ComfyUI API 工作流
import json
WF_JSON = '{"92":{"inputs":{"filename_prefix":"video/MiniMax_H3","format":"auto","format.codec":"auto","codec":"auto","video":["138",0]},"class_type":"SaveVideo","_meta":{"title":"output_video"}},"114":{"inputs":{"image":"h3_first.png"},"class_type":"LoadImage","_meta":{"title":"first_frame"}},"121":{"inputs":{"value":960},"class_type":"PrimitiveInt","_meta":{"title":"width"}},"122":{"inputs":{"value":544},"class_type":"PrimitiveInt","_meta":{"title":"height"}},"123":{"inputs":{"value":5},"class_type":"PrimitiveInt","_meta":{"title":"duration"}},"124":{"inputs":{"image":"h3_last.png"},"class_type":"LoadImage","_meta":{"title":"last_frame"}},"125":{"inputs":{"value":12345},"class_type":"PrimitiveInt","_meta":{"title":"seed"}},"126":{"inputs":{"value":"A luminous blue sphere glides smoothly across the dark frame as the camera slowly pushes in. Soft particles drift and warm highlights move across the background. Native stereo audio: a gentle cinematic whoosh and low ambient tone, no speech."},"class_type":"PrimitiveStringMultiline","_meta":{"title":"prompt"}},"127":{"inputs":{"vae_name":"minimax_h3_video_vae_fp16.safetensors"},"class_type":"VAELoader","_meta":{"title":"加载VAE"}},"128":{"inputs":{"vae_name":"minimax_h3_audio_vae_fp32.safetensors"},"class_type":"VAELoader","_meta":{"title":"加载VAE"}},"129":{"inputs":{"samples":["133",0],"vae":["128",0]},"class_type":"VAEDecodeAudio","_meta":{"title":"VAE解码（音频）"}},"130":{"inputs":{"samples":["133",0],"vae":["127",0]},"class_type":"VAEDecode","_meta":{"title":"VAE解码"}},"131":{"inputs":{"sampler_name":"res_multistep"},"class_type":"KSamplerSelect","_meta":{"title":"K采样器选择"}},"132":{"inputs":{"scheduler":"simple","steps":4,"denoise":1,"model":["142",0]},"class_type":"BasicScheduler","_meta":{"title":"基本调度器"}},"133":{"inputs":{"noise":["137",0],"guider":["134",0],"sampler":["131",0],"sigmas":["132",0],"latent_image":["139",1]},"class_type":"SamplerCustomAdvanced","_meta":{"title":"自定义采样器（高级）"}},"134":{"inputs":{"model":["142",0],"conditioning":["139",0]},"class_type":"BasicGuider","_meta":{"title":"基本引导器"}},"135":{"inputs":{"unet_name":"minimax_h3_fl2va_pruned_int8_convrot.safetensors","weight_dtype":"default"},"class_type":"UNETLoader","_meta":{"title":"select_models_diffusion_models"}},"136":{"inputs":{"clip_name":"qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors","type":"minimax","device":"default"},"class_type":"CLIPLoader","_meta":{"title":"加载CLIP"}},"137":{"inputs":{"noise_seed":["125",0]},"class_type":"RandomNoise","_meta":{"title":"随机噪波"}},"138":{"inputs":{"fps":24,"bit_depth":8,"color_space":"sRGB","codec":"none","images":["130",0],"audio":["129",0]},"class_type":"CreateVideo","_meta":{"title":"创建视频"}},"139":{"inputs":{"prompt":["126",0],"width":["121",0],"height":["122",0],"length":["140",1],"clip":["136",0],"vae":["127",0],"first_frame":["114",0],"last_frame":["124",0]},"class_type":"MiniMaxH3ImageToVideo","_meta":{"title":"MiniMax H3 图像转视频"}},"140":{"inputs":{"expression":"max(5, round(a * 24)) + (5 - (max(5, round(a * 24)) % 17)) % 17","values.a":["123",0]},"class_type":"ComfyMathExpression","_meta":{"title":"数学表达式"}},"142":{"inputs":{"lora_name":"minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors","strength_model":1,"model":["135",0]},"class_type":"LoraLoaderModelOnly","_meta":{"title":"LoRA加载器（仅模型）"}}}'
print("工作流节点：", len(json.loads(WF_JSON)))


In [ ]:
# 单元 8：REST 客户端、参数映射、5 秒工作流模板（无 GPU 推理）
import json, os, time, subprocess, uuid, requests

CLIENT_ID = str(uuid.uuid4())

def slot(wf, cls, title):
    for v in wf.values():
        if v.get("class_type") == cls and v.get("_meta", {}).get("title") == title:
            return v["inputs"]
    raise KeyError("%s / %s" % (cls, title))

def prepare(wf, prompt_text, dur, width, height, seed, first, last):
    slot(wf, "PrimitiveStringMultiline", "prompt")["value"] = prompt_text
    slot(wf, "PrimitiveInt", "duration")["value"] = dur
    slot(wf, "PrimitiveInt", "width")["value"] = width
    slot(wf, "PrimitiveInt", "height")["value"] = height
    slot(wf, "PrimitiveInt", "seed")["value"] = seed
    slot(wf, "LoadImage", "first_frame")["image"] = first
    slot(wf, "LoadImage", "last_frame")["image"] = last
    return wf

def pick_video(entry):
    for _, out in (entry.get("outputs") or {}).items():
        for key in ("images", "gifs", "videos"):
            for item in (out.get(key) or []):
                if str(item.get("filename", "")).lower().endswith((".mp4", ".webm", ".mov")):
                    return item
    return None

def run_job(wf, tag, timeout=3600):
    t_submit = time.time()
    r = requests.post(BASE + "/prompt", json={"prompt": wf, "client_id": CLIENT_ID}, timeout=60)
    if r.status_code != 200:
        print("[失败] /prompt 返回", r.status_code)
        err = r.json()
        print("node_errors:", json.dumps(err.get("node_errors", {}), ensure_ascii=False)[:2000])
        raise SystemExit("提交失败，先看上面的 node_errors")

    pid = r.json()["prompt_id"]
    print("已提交 prompt_id =", pid, "| 模型加载+生成可能要十几分钟，耐心等")

    t_gen = None
    last = -1
    while time.time() - t_submit < timeout:
        h = requests.get(BASE + "/history/" + pid, timeout=30).json()
        if pid in h:
            entry = h[pid]
            status = (entry.get("status") or {}).get("status_str")
            if status == "success":
                t_gen = time.time() - t_submit
                break
            if status == "error":
                print("执行出错:", json.dumps((entry.get("status") or {}).get("messages", []),
                                              ensure_ascii=False)[:2000])
                raise SystemExit("工作流执行失败")
        el = int(time.time() - t_submit)
        if el // 30 != last:
            last = el // 30
            try:
                q = requests.get(BASE + "/queue", timeout=10).json()
                print("  %4ds  排队 %d / 运行 %d" % (el, len(q.get("queue_pending", [])),
                                                    len(q.get("queue_running", []))))
            except Exception:
                print("  %4ds" % el)
        time.sleep(3)
    else:
        raise SystemExit("超时未完成")

    item = pick_video(entry)
    if not item:
        print("outputs 原文:", json.dumps(entry.get("outputs", {}), ensure_ascii=False)[:1500])
        raise SystemExit("完成但没有视频产物")

    q = {"filename": item["filename"], "subfolder": item.get("subfolder", ""), "type": item.get("type", "output")}
    view = requests.get(BASE + "/view", params=q, timeout=600)
    view.raise_for_status()
    data = view.content
    out = os.path.join(OUT_DIR, "%s_%s" % (tag, item["filename"]))
    with open(out, "wb") as fh:
        fh.write(data)
    print("已保存:", out, "%.1f MB" % (len(data) / 2**20))
    print("生成耗时: %.1f 秒 (含模型加载)" % t_gen)
    return out, t_gen

wf5 = prepare(json.loads(WF_JSON), "A luminous blue sphere glides across the dark frame. Native stereo audio: a gentle cinematic whoosh and low ambient tone.",
              5, 960, 544, 12345, n1, n2)


In [ ]:
# 可选：5 秒回归测试；若只需 10 秒，跳过此格及下一格视频核验
mp4_5s, t5 = run_job(wf5, "test5s")


In [ ]:
# 可选：仅在运行 5 秒测试后核验其产物
import json, subprocess

def probe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-print_format", "json",
                        "-show_format", "-show_streams", path],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("ffprobe 不可用:", r.stderr[-300:])
        return None
    return json.loads(r.stdout)

info = probe(mp4_5s)
if info:
    v = next((s for s in info["streams"] if s["codec_type"] == "video"), None)
    a = next((s for s in info["streams"] if s["codec_type"] == "audio"), None)
    print("分辨率 :", v.get("width"), "x", v.get("height"))
    fr = v.get("avg_frame_rate", "0/1").split("/")
    print("帧率   : %.2f fps" % (float(fr[0]) / float(fr[1] or 1)))
    print("帧数   :", v.get("nb_frames"))
    print("时长   : %.2f 秒" % float(info["format"]["duration"]))
    print("音轨   :", "有 (%s)" % a.get("codec_name") if a else "无  <-- 异常，H3 应带音频")
    print("\n判定：有音轨 + 24fps + 帧数≈124 + 时长≈5.17s = 真实 H3 推理输出")

In [ ]:
# ============ 单元 10：10 秒 720P 目标测试 ============
wf10 = prepare(json.loads(WF_JSON),
               "A luminous blue sphere glides across the dark frame. Native stereo audio: a gentle cinematic whoosh and low ambient tone.",
               10, 1344, 768, 12345, n1, n2)
mp4_10s, t10 = run_job(wf10, "target10s", timeout=5400)

In [ ]:
# 单元 12：备份工作流与成片；按需备份模型（完整文件跳过）
import os, shutil, subprocess
if DRIVE_ROOT:
    for sub in ("outputs", "workflows", "models"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
    for name in os.listdir(OUT_DIR):
        src, dst = os.path.join(OUT_DIR, name), os.path.join(DRIVE_ROOT, "outputs", name)
        if os.path.isfile(src) and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src)):
            shutil.copy2(src, dst)
    for name, wf in (("h3_5s_api.json", json.dumps(wf5, ensure_ascii=False, indent=2)), ("h3_10s_720p_api.json", json.dumps(wf10, ensure_ascii=False, indent=2))):
        dst = os.path.join(DRIVE_ROOT, "workflows", name)
        with open(dst, "w", encoding="utf-8") as f:
            f.write(wf)
    if shutil.disk_usage("/content/drive/MyDrive").free > 45 * 2**30:
        subprocess.run(["rsync", "-rt", "--size-only", "--partial-dir=.rsync-partial", "--exclude=.cache/", MODELS_DIR + "/", os.path.join(DRIVE_ROOT, "models") + "/"], check=True)
        print("权重已备份；下次优先从 Drive 恢复。")
    else:
        print("Drive 空间不足，已保留成片与工作流，跳过权重备份。")
else:
    print("Drive 未挂载，成片保留在", OUT_DIR)


In [ ]:
# ============ 单元 11：会话摘要 + CCU 记录 ============
import csv, datetime, os, subprocess, json

def gpu_name():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return "unknown"

row = {
    "时间": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "GPU": gpu_name(),
    "工作流": "minimax_h3_fl2va_pruned_int8_convrot + turbo4step",
    "5s耗时秒": round(t5, 1) if "t5" in dir() else "",
    "10s耗时秒": round(t10, 1) if "t10" in dir() else "",
    "CCU消耗": "",   # 需在 Colab 页面「资源」里查看后手工填入
    "备注": "",
}
csv_path = os.path.join(LOG_DIR, "ccu_log.csv")
new = not os.path.exists(csv_path)
with open(csv_path, "a", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    if new:
        w.writeheader()
    w.writerow(row)

print("已记录到", csv_path)
print()
print("产物目录:", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR))[-5:]:
    print("  ", f)
print()
print("CCU 请到 Colab 右上角「资源 / Resources」查看本次实际消耗，手工填进 ccu_log.csv。")

## Mac 无限画布连接状态

本 Notebook 已有真实 H3 图生视频与原生音频输出记录。官方 Colab CLI 提供 `colab ssh --proxy-mode`，可以尝试将 Mac 本地端口转发到运行时内的 ComfyUI；仓库脚本只允许连接已有 CLI 会话。本地隧道、跨源 REST 与 WebSocket 101 已实测；画布生成由用户报告成功，视频自动回传的文件规格与音轨尚未独立核验。不要向公网开放未认证的 8188 端口。


In [ ]:
# ============ 排障：查看 ComfyUI 日志尾部 ============
import os
p = os.path.join(LOG_DIR, "comfyui.log")
if os.path.exists(p):
    print(open(p, encoding="utf-8", errors="replace").read()[-4000:])
else:
    print("还没有日志")